In [53]:
#!/usr/bin/env python3
import os
import glob
import pyterrier as pt
import polars as pl
import pandas as pd

In [27]:
if not pt.started():
    pt.init()  # downloads Terrier jars, needs Java installed

/var/folders/yr/ljjqmpj92wncw9wrv495wq300000gn/T/ipykernel_43873/1491358899.py:1: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


In [34]:
PARQUET_PATH = "../output/dicty_gold_build/3_articles_cleaned_abstract.parquet"   
INDEX_PATH = os.path.abspath("../indexes/terrier_index_dicty_22.01.26")              

In [35]:
os.makedirs(INDEX_PATH, exist_ok=True)

In [36]:
def iter_docs_from_parquet(parquet_path: str):
    # keep the same script logic: dedup by pmid (keep last), and concatenate title + abstract
    df = (
        pl.read_parquet(parquet_path)
        .select(["pmid", "title", "abstract_clean"])
        .with_columns([
            pl.col("pmid").cast(pl.Utf8),
            pl.col("title").fill_null(""),
            pl.col("abstract_clean").fill_null(""),
        ])
    )

    latest = {}
    for pmid, title, abstract in df.iter_rows():
        pmid = (pmid or "").strip()
        if not pmid:
            continue
        latest[pmid] = {
            "docno": pmid,
            "text": f"{title} {abstract}".strip()
        }

    yield from latest.values()

In [46]:
# build index
indexer = pt.IterDictIndexer(
    INDEX_PATH,
    text_attrs=["text"],
    meta={"docno": 32, "text": 20000},  # ✅ allow long title+abstract
    overwrite=True,
    threads=1
)

In [47]:
# quick sanity:
first = next(iter_docs_from_parquet(PARQUET_PATH))
print(first.keys(), first["docno"][:10], first["text"][:80])

dict_keys(['docno', 'text']) 2654141 Centrin-mediated microtubule severing during flagellar excision in Chlamydomonas


In [48]:
index_ref = indexer.index(iter_docs_from_parquet(PARQUET_PATH))
print("OK indexed at:", INDEX_PATH)

OK indexed at: /Users/yun/develop/dictycite/indexes/terrier_index_dicty_22.01.26


In [52]:
# check
index = pt.IndexFactory.of(index_ref)     # or pt.IndexFactory.of(INDEX_PATH)
coll_stats = index.getCollectionStatistics()

print("num_docs:", coll_stats.getNumberOfDocuments())
print("num_terms:", coll_stats.getNumberOfUniqueTerms())
print("num_tokens:", coll_stats.getNumberOfTokens())

num_docs: 20447
num_terms: 51101
num_tokens: 2622211


In [54]:
# try query
bm25 = pt.terrier.Retriever(index, wmodel="BM25")
q = pd.DataFrame([{"qid":"q1", "query":"A recently reconstructed spatially fourth and temporally second order accurate, implicit, stable high order compact scheme has been employed to carry out simulations of the Oregonator model of excitable media."}])
bm25.transform(q).head(5)

,qid,docid,docno,rank,score,query
0,q1,13585,36008513,0,77.109085,A recently reconstructed spatially fourth and ...
1,q1,13194,12513360,1,21.287180,A recently reconstructed spatially fourth and ...
2,q1,8067,34260581,2,20.649156,A recently reconstructed spatially fourth and ...
3,q1,13,20689723,3,19.947240,A recently reconstructed spatially fourth and ...
4,q1,17050,20864631,4,18.740571,A recently reconstructed spatially fourth and ...


In [56]:
res = bm25.transform(q)
top_pmid = str(res.loc[0, "docno"])   # res is your BM25 results df (pandas)
df = pl.read_parquet(PARQUET_PATH)

row = (
    df.with_columns(pl.col("pmid").cast(pl.Utf8))
      .filter(pl.col("pmid") == top_pmid)
      .select(["pmid", "title", "abstract_clean"])
)

row

pmid,title,abstract_clean
str,str,str
"""36008513""","""Influence of a circular obstac…","""The current study envisages to…"


results looks good!